In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from PIL import Image
from datetime import datetime
import shutil

In [2]:
SOURCE_DIR = Path("../data/external/mit_indoor/Images/")
TARGET_DIR = Path("../data/raw/")
OUTPUT_CSV = Path("../data/processed/image_index.csv")

In [5]:
rows = []

for img_path in SOURCE_DIR.rglob("*"):
    if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".webp"]:
        continue
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            width, height = img.size

            label = img_path.parent.name
            dataset = img_path.parts[-3] if len(img_path.parts) >= 3 else "unknown"

            target_folder = TARGET_DIR / dataset / label
            target_folder.mkdir(parents=True, exist_ok=True)

            target_path = target_folder / (img_path.stem + ".jpg")
            img.save(target_path, format="JPEG", quality=95)

            md5 = hashlib.md5(target_path.read_bytes()).hexdigest()

            rows.append({
                "image_id": target_path.stem,
                "source_dataset": dataset,
                "original_path": str(img_path),
                "new_path": str(target_path),
                "label": label,
                "width": width,
                "height": height,
                "format": "jpg",
                "file_size_kb": round(target_path.stat().st_size / 1024, 2),
                "hash_md5": md5,
                "ingestion_time": datetime.utcnow().isoformat()
            })
    except Exception:
        continue

pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)